In [2]:
import math 
import pandas as pd
import numpy as np
#import setup
from docplex.mp.model import Model
from docplex.mp.solution import SolveSolution
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.cluster import KMeans
import random
import sys

np.random.seed(2657)

In [3]:
def findOptimalSolution4(compactPackageList, compactPackageDependency, DuplicateRowNumberList, HasanClusters):
    
    mdl = Model('MIP')
    
    global PackageList, OldCluster, OldClusterList, PackageDependency

    nPackagesRow = len(DuplicateRowNumberList)
    nPackages = len(PackageList)
    nClusters = 20#len(OldClusterList)
    nSubclusters = len(HasanClusters)

    print(nPackages, nClusters)
    
    y = mdl.binary_var_dict([(i,k) for i in range(nPackagesRow) for k in range(nClusters)], name='y') #i. paket, k in clsutersi cagiyor mu?
    z = mdl.integer_var_dict([(k) for k in range(nClusters)], name='z') #k. cluster cagiriliyor mu
    x = mdl.binary_var_dict([(l,k) for l in range(nSubclusters) for k in range(nClusters)], name='x') #j. subcluster, k in clsuterda mi?
    P = mdl.continuous_var_dict([(i,k) for i in range(nPackagesRow) for k in range(nClusters)], name='P') #i. paket, k in clsuter kac paket cagiyor
    SP = mdl.continuous_var_dict([(i) for i in range(nPackagesRow)], name='SP') #i. paketin cagirdigi toplam paket sayisi
    
    #mdl.minimize(mdl.sum(P[i,k] for i in range(nPackagesRow) for k in range(nClusters)))
    mdl.minimize(mdl.sum(DuplicateRowNumberList[i]*P[i,k] for i in range(nPackagesRow) for k in range(nClusters)))
    #mdl.minimize(mdl.sum(z[k] for k in range(nClusters)))
        
    #Constraints
    #mdl.add_constraints(PackageDependency[i][j]+x[j,k]-y[i,k] <= 1 for i in range(nPackages) for j in range(nPackages) for k in range(nClusters)) 
    
    mdl.add_constraints(mdl.sum(x[l,k]*compactPackageDependency[i][j] for l in range(nSubclusters) for j in HasanClusters[l]) <= 1000*y[i,k] for i in range(nPackagesRow) for k in range(nClusters)) 
    mdl.add_constraints(mdl.sum(x[l,k] for k in range(nClusters)) == 1 for l in range(nSubclusters)) 
    mdl.add_constraints(mdl.sum(DuplicateRowNumberList[i]*y[i,k] for i in range(nPackagesRow)) - z[k] <= 0 for k in range(nClusters)) 
    mdl.add_constraints(mdl.sum(x[l,k]*len(HasanClusters[l]) for l in range(nSubclusters)) <= P[i,k]+1000*(1-y[i,k]) for i in range(nPackagesRow) for k in range(nClusters)) 
    #mdl.add_constraint(mdl.sum(P[i,k] for i in range(nPackages) for k in range(nClusters))<= 1396) 
    mdl.add_constraints(z[k+1] - z[k] <= 0 for k in range(nClusters-1)) #symmetry breaking
    mdl.add_constraints(z[k+1] - z[k] <= 0 for k in range(nClusters-1)) 
    mdl.add_constraint(mdl.sum(z[k] for k in range(nClusters)) <= 2082) 
    
    mdl.parameters.timelimit=10800 #time limit

    solution = mdl.solve(log_output= True)
    #print(solution.solve_status) #if it says feasible, it is not optimal
    obj = solution.get_objective_value()
    mdl.solve_details
    print(obj)
    
    XSol = [x[l,k].solution_value for l in range(nSubclusters) for k in range(nClusters)]   
    ZSol = [z[k].solution_value for k in range(nClusters)] 

    data = np.array(XSol)
    shape = (nSubclusters, nClusters)
    data = data.reshape(shape)
    data = data.T
    data = data.astype(int)
    data = data[~np.all(data == 0, axis=1)]

    # Determine the total number of packages
    num_packages = max(max(pkg_list) for pkg_list in HasanClusters) + 1
    
    # Initialize the new matrix (clusters x packages) with zeros
    packages_matrix = np.zeros((data.shape[0], num_packages), dtype=int)
    
    # Fill the packages_matrix
    for set_index, package_indices in enumerate(HasanClusters):
        for package in package_indices:
            packages_matrix[:, package] |= data[:, set_index]


    ZSol = np.array(ZSol)
    ZSol = ZSol.astype(int)
    # shape = (nCodes, nCodes)
    # data2 = data2.reshape(shape)
    selected_indices = np.where(ZSol == 1)[0]
    selected_indices = selected_indices.tolist()
    
    mdl.end
    
    return packages_matrix, selected_indices

  

In [4]:
def count_cluster_calls(package_dependency_matrix, cluster_matrix, package_list):
    # Number of clusters
    num_clusters = cluster_matrix.shape[0]
    
    # Initialize a vector to count calls for each cluster
    cluster_call_count = np.zeros(num_clusters, dtype=int)
    
    # For each package in the package list, get the required clusters
    unique_clusters_selected = set()
    for package in package_list:
        required_clusters = get_required_clusters(package_dependency_matrix, cluster_matrix, package)
        
        # Increment the call count for each required cluster and record unique clusters
        for cluster in required_clusters:
            cluster_call_count[cluster] += 1
            unique_clusters_selected.add(cluster)
    
    # Calculate the total unique clusters selected
    total_unique_clusters_selected = len(unique_clusters_selected)
    
    # Calculate the number of packages per cluster
    packages_per_cluster = cluster_matrix.sum(axis=1)
    
    # Calculate the total packages called as the sum product
    total_packages_called = np.dot(cluster_call_count, packages_per_cluster)
    
    return cluster_call_count, total_unique_clusters_selected, total_packages_called


In [5]:
def get_required_clusters(package_dependency_matrix, cluster_matrix, p):
    # Step 1: Find the dependencies of package p
    package_dependencies = np.where(package_dependency_matrix[p] == 1)[0]
    
    # Step 2: Initialize an empty set to store unique cluster indices
    required_clusters = set()
    
    # Step 3: For each dependency package, find the clusters that include it
    for package in package_dependencies:
        clusters_with_package = np.where(cluster_matrix[:, package] == 1)[0]
        required_clusters.update(clusters_with_package)
    
    # Return the unique clusters as a sorted list
    return sorted(required_clusters)

In [6]:
def remove_duplicate_rows(PackageList, PackageDependency):
    PackageDependency = np.array(PackageDependency)  # Convert to numpy array for easier manipulation
    n = len(PackageDependency)
    duplicates = []
    to_remove = set()
    DuplicateRowNumberList = []
    DuplicateRowList = []
    
    # Dictionary to map unique rows to their original indices
    unique_row_indices = {}

    # Find duplicate rows
    for i in range(n):
        if i in to_remove:
            continue
        # Initialize the list to track duplicates for the current package
        duplicate_indices = [i]
        duplicate_count = 1
        
        for j in range(i + 1, n):
            if np.array_equal(PackageDependency[i], PackageDependency[j]):
                duplicates.append((i, j))
                to_remove.add(j)
                duplicate_indices.append(j)
                duplicate_count += 1

        # Update DuplicateRowNumberList and DuplicateRowList
        DuplicateRowNumberList.append(duplicate_count)
        DuplicateRowList.append(duplicate_indices)

    # Print duplicate pairs
    # print("Duplicate Pairs:")
    # for (i, j) in duplicates:
    #     print(f"Duplicate pair: Index {i} ({PackageList[i]}) and Index {j} ({PackageList[j]})")

    # Remove duplicates from both PackageDependency and PackageList
    PackageDependency = [row for idx, row in enumerate(PackageDependency) if idx not in to_remove]
    PackageList = [pkg for idx, pkg in enumerate(PackageList) if idx not in to_remove]

    return PackageList, PackageDependency, DuplicateRowNumberList, DuplicateRowList

In [7]:
def check_clusters_adherence(PackageList, index_array, OldClusters):
    # Initialize a set to store violations
    violations = set()

    # Iterate through the index_array to check for adherence
    for cluster_indices in index_array:
        # Iterate over all pairs of indices in the cluster
        for i in range(len(cluster_indices)):
            for j in range(i + 1, len(cluster_indices)):
                package_a = cluster_indices[i]
                package_b = cluster_indices[j]

                # Check if both packages are in the same row (cluster) in OldClusters
                in_same_cluster = any(
                    OldClusters[row][package_a] == 1 and OldClusters[row][package_b] == 1
                    for row in range(OldClusters.shape[0])
                )

                # If not in the same cluster, record the pair
                if not in_same_cluster:
                    violations.add((PackageList[package_a], PackageList[package_b]))

    # Output violations
    if violations:
        print("The following package pairs should be in the same cluster but are not:")
        for package_a, package_b in violations:
            print(f" - {package_a} and {package_b}")
    else:
        print("All package pairs adhere to the clustering rules.")

In [11]:
def readdata(dependencyFile, inputFile, clusterFile):

    global PackageList, PackageListCheck,  OldClusterList, PackageDependency, OldClusters, HasanClusters

    PackageList = []
    PackageListCheck = []
    OldClusterList = []

    PackageDependency = []
    OldClusters = []

    
    with open(inputFile) as f:
        lines = f.readlines()
        
    for i in lines:
        strList = i.split()
        OldClusterList.append(strList[1])
        PackageList.append(strList[2])

    PackageList = list(set(PackageList))
    OldClusterList = list(set(OldClusterList))

    with open(dependencyFile) as f:
        lines = f.readlines()
        
    for i in lines:
        strList = i.split()
        PackageListCheck.append(strList[1])
        PackageListCheck.append(strList[2])

    PackageListCheck = list(set(PackageListCheck))

    print(set(PackageListCheck).issubset(set(PackageListCheck))) 


    PackageDependency = [ [0]*len(PackageList) for i in range(len(PackageList))]
    with open(dependencyFile) as f:
        lines = f.readlines()
    for i in lines:
        strList = i.split()        
        index1 = PackageList.index(strList[1])
        index2 = PackageList.index(strList[2])
        PackageDependency[index1][index2] = 1

    OldClusters = [ [0]*len(PackageList) for i in range(len(OldClusterList))]
    with open(inputFile) as f:
        lines = f.readlines()
    for i in lines:
        strList = i.split()        
        index1 = OldClusterList.index(strList[1])
        index2 = PackageList.index(strList[2])
        OldClusters[index1][index2] = 1    

    clusters = {}

    # Open the file and process line by line
    with open(clusterFile, 'r') as file:
        for line in file:
            # Split the line into parts
            parts = line.strip().split()
            
            # Extract cluster name and package name
            cluster_name = parts[1]  # Second word is the cluster name
            package_name = ' '.join(parts[2:])  # Remaining words are the package name
            
            # Add package name to the corresponding cluster
            if cluster_name not in clusters:
                clusters[cluster_name] = []
            clusters[cluster_name].append(package_name)
    
    # Convert the dictionary values to a list of lists
    HasanClustersTemp = list(clusters.values())


    package_index_map = {package: index for index, package in enumerate(PackageList)}

    # Create the new array with indices instead of package names
    HasanClusters = []
    for cluster in HasanClustersTemp:
        cluster_indices = [package_index_map[package] for package in cluster]
        HasanClusters.append(cluster_indices)
    
    
    

In [12]:
readdata('java_packages_packages_dependencies_cleaned.rsf', 'java_modules_packages_dependencies_cleaned.rsf', 'clustering_primer.rsf') #(Manuel kümeleme)
PackageDependency = np.array(PackageDependency)
OldClusters = np.array(OldClusters)

True


In [13]:
# Check adherence
check_clusters_adherence(PackageList, HasanClusters, OldClusters)

The following package pairs should be in the same cluster but are not:
 - com.sun.xml.internal.ws.protocol.xml and javax.rmi.ssl
 - sun.awt.shell and java.util.regex
 - com.sun.jmx.snmp.daemon and javax.management.openmbean
 - com.oracle.webservices.internal.api.message and java.util.concurrent.locks
 - com.sun.istack.internal and com.sun.xml.internal.ws.api.pipe.helper
 - com.sun.corba.se.spi.protocol and java.rmi
 - com.sun.corba.se.impl.naming.namingutil and java.rmi
 - javax.xml.namespace and javax.xml.soap
 - com.sun.swing.internal.plaf.metal.resources and com.sun.xml.internal.ws.assembler.dev
 - com.sun.jndi.toolkit.ctx and com.sun.security.sasl.util
 - com.sun.xml.internal.bind.v2.model.runtime and com.sun.xml.internal.ws.spi
 - java.text and jdk.net
 - javax.xml.transform and com.sun.xml.internal.ws.message.source
 - com.sun.swing.internal.plaf.basic.resources and com.sun.xml.internal.ws.wsdl.writer.document.http
 - com.sun.xml.internal.ws.policy and javax.management.timer
 - s

In [14]:
package_indices  = list(range(len(PackageList)))

cluster_call_count, total_unique_clusters_selected, total_packages_called = count_cluster_calls(
    PackageDependency, OldClusters, package_indices)

print("Cluster Call Count:", cluster_call_count)
print("Total Call Count:", sum(cluster_call_count))
print("Total Unique Clusters Selected:", total_unique_clusters_selected)
print("Total Packages Called:", total_packages_called)

Cluster Call Count: [ 38   7   6   4 819  32   9   4   2   2   1   8  37   4   1   5   6  36
  84  19   2   4   2   2  15   1   3   7 129 124 278 122   2   4   2 109
   3   1  30  12   5   4 110   3]
Total Call Count: 2098
Total Unique Clusters Selected: 44
Total Packages Called: 191252


In [15]:
compactPackageList, compactPackageDependency, DuplicateRowNumberList, DuplicateRowList = remove_duplicate_rows(PackageList, PackageDependency)

In [16]:
Clusters, selected_clusters = findOptimalSolution4(compactPackageList, compactPackageDependency, DuplicateRowNumberList,HasanClusters)

832 20
Using size restricted mode (Could not find directory for cpxchecklic).
CPLEX Error  1016: Community Edition. Problem size limits exceeded. Purchase at http://ibm.biz/error1016.


DOcplexLimitsExceeded: **** Promotional version. Problem size limits (1000 vars, 1000 consts) exceeded, model has 34196 vars, 31976 consts, CPLEX code=1016

In [ ]:
cluster_call_count, total_unique_clusters_selected, total_packages_called = count_cluster_calls(
    PackageDependency, Clusters, package_indices)

print("Cluster Call Count:", cluster_call_count)
print("Total Call Count:", sum(cluster_call_count))
print("Total Unique Clusters Selected:", total_unique_clusters_selected)
print("Total Packages Called:", total_packages_called)

In [ ]:
# Open a file to write the output
with open("clusters.txt", "w") as file:
    # Iterate over each cluster (row in the matrix)
    for i, cluster in enumerate(Clusters):
        cluster_name = f"Cluster{i + 1}"
        # Iterate over each module (column in the matrix)
        for j, is_in_cluster in enumerate(cluster):
            if is_in_cluster == 1:
                module_name = PackageList[j]
                # Write the information to the file
                file.write(f"contains {cluster_name} {module_name}\n")